# Exercise 4 — select_mcp_tool

`build_mcp_selection_prompt` renders the client's tool list into a router prompt.  `select_mcp_tool` calls the LLM, parses the JSON response, validates the tool name, and falls back to `"none"` if anything goes wrong.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class MCPToolDef:
    name: str
    description: str
    input_schema: dict = field(default_factory=dict)

def tool_schema_text(tools):
    lines = []
    for t in tools:
        params = ", ".join(t.input_schema.keys())
        lines.append("- " + t.name + "(" + params + "): " + t.description)
    return "\n".join(lines)
class MCPServer:
    def __init__(self, name="mcp_server"):
        self.name = name
        self._tools = {}
    def tool(self, name, description, schema=None):
        def _decorator(fn):
            self._tools[name] = {"def": MCPToolDef(name, description, schema or {}), "fn": fn}
            return fn
        return _decorator
    def list_tools(self):
        return [e["def"] for e in self._tools.values()]
    def call_tool(self, name, args):
        e = self._tools.get(name)
        if e is None:
            return "Error: unknown tool " + repr(name)
        try:
            return str(e["fn"](**args))
        except Exception as exc:
            return "Error: " + str(exc)
def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    start = str(text).find("{")
    end   = str(text).rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None
class MCPClient:
    def __init__(self, server=None, tool_call_fn=None):
        self._server = server
        self._tool_call_fn = tool_call_fn
    def list_tools(self):
        if self._server is not None:
            return self._server.list_tools()
        return []
    def call_tool(self, name, args):
        if self._tool_call_fn is not None:
            return self._tool_call_fn(name, args)
        if self._server is not None:
            return self._server.call_tool(name, args)
        return "Error: no server or tool_call_fn configured"

# ── Exercise: implement selection functions ──────────────────────────────────

def build_mcp_selection_prompt(query, tools):
    # TODO: call tool_schema_text(tools) to get the menu string
    # Build a system message that lists the tools and asks for JSON with
    # "tool" and "args" keys; include the fallback to "none"
    # Return [{"role": "system", ...}, {"role": "user", "content": "Request: " + str(query)}]
    return [{"role": "system", "content": ""}, {"role": "user", "content": str(query)}]


def select_mcp_tool(query, client, llm_fn=None):
    # TODO: get tools from client.list_tools()
    # If no tools, return {"tool": "none", "args": {}}
    # Build prompt, call LLM, safe_parse_json the response
    # Validate: if name not in known tool names, set name = "none"
    # Return {"tool": name, "args": args_dict}
    return {"tool": "none", "args": {}}


### Checks

In [ ]:
import json

def _mock_llm(tool, args=None):
    payload = json.dumps({"tool": tool, "args": args or {}})
    return lambda messages: payload

checks = 0

# helper server
_srv = MCPServer("t")
@_srv.tool("wc", "Count words.", {"text": "str"})
def _wc(text): return str(len(str(text).split()))
_client = MCPClient(server=_srv)

# 1 — select_mcp_tool returns dict with "tool" and "args"
try:
    sel = select_mcp_tool("count words", _client, llm_fn=_mock_llm("wc", {"text": "hi"}))
    assert "tool" in sel and "args" in sel
    checks += 1; print("✅ 1 select_mcp_tool returns {tool, args}")
except Exception as e:
    print("❌ 1:", e)

# 2 — correct tool selected
try:
    sel = select_mcp_tool("how many words?", _client, llm_fn=_mock_llm("wc", {"text": "hello world"}))
    assert sel["tool"] == "wc" and sel["args"] == {"text": "hello world"}
    checks += 1; print("✅ 2 correct tool and args returned")
except Exception as e:
    print("❌ 2:", e)

# 3 — unknown tool name -> "none"
try:
    sel = select_mcp_tool("something", _client, llm_fn=_mock_llm("nonexistent"))
    assert sel["tool"] == "none"
    checks += 1; print("✅ 3 unknown tool name coerced to 'none'")
except Exception as e:
    print("❌ 3:", e)

# 4 — build_mcp_selection_prompt includes tool names
try:
    tools = _client.list_tools()
    prompt = build_mcp_selection_prompt("test", tools)
    combined = " ".join(m.get("content", "") for m in prompt)
    assert "wc" in combined
    checks += 1; print("✅ 4 build_mcp_selection_prompt includes tool names")
except Exception as e:
    print("❌ 4:", e)

# 5 — empty client returns "none" without calling LLM
try:
    empty_client = MCPClient()
    sel = select_mcp_tool("anything", empty_client, llm_fn=None)
    assert sel["tool"] == "none"
    checks += 1; print("✅ 5 empty client returns 'none' immediately")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
